In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy
import os
from collections import Counter
from tqdm import trange

# Define file paths
csv_base_path = "G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\Edge_adjunct_LC\\Ecoregion_Classification\\county_{}.csv"
shapefile_path = "G:\\Hangkai\\CONUS Forest Edge Mapping\\CONUS shapefile\\cb_2018_us_county_5m.shp"
output_dir = "adjunct_landcover_maps"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Read the Shapefile
county_shapefile = gpd.read_file(shapefile_path)

# Convert key columns to string to ensure proper merging
county_shapefile['STATEFP'] = county_shapefile['STATEFP'].astype(str).str.zfill(2)
county_shapefile['COUNTYFP'] = county_shapefile['COUNTYFP'].astype(str).str.zfill(3)

# Define land cover mapping
land_cover_mapping = {
    '1': 'Developed',
    '2': 'Cropland',
    '3': 'Grass/Shrub',
    '0': 'Tree Cover',  # Recoded from '4'
    '5': 'Water',
    '6': 'Wetland',
    '7': 'Ice/Snow',
    '8': 'Barren'
}

def decode_landcover(code):
    """ Decode the five-digit landcover code into readable labels for each side. """
    code_str = str(int(code)).zfill(5)
    return [
        land_cover_mapping.get(digit, 'Unknown') for digit in code_str[1:] if digit != '0'
    ]

# Function to determine the major land cover type
def major_land_cover(row):
    counter = Counter()
    for col in row.index:
        if col not in ['STATEFP', 'NAME', 'COUNTYFP']:
            land_covers = decode_landcover(col)
            pixel_sum = row[col]
            land_cover_count = Counter(land_covers)
            for land_cover, count in land_cover_count.items():
                if land_cover != 'Tree Cover':
                    counter[land_cover] += pixel_sum * count
    if counter:
        return max(counter, key=counter.get)
    return 'Unknown'

# Loop through each year's data and generate maps
for year in trange(1985, 2022):
    print(f'Processing year: {year}')
    csv_path = csv_base_path.format(year)
    
    if os.path.exists(csv_path):
        data = pd.read_csv(csv_path)

        # Convert key columns to string to ensure proper merging
        data['STATEFP'] = data['STATEFP'].astype(str).str.zfill(2)
        data['COUNTYFP'] = data['COUNTYFP'].astype(str).str.zfill(3)

        # Fill NaN values with 0
        data = data.fillna(0)

        # Decode the landcover and find the major landcover type for each county
        data['major_land_cover'] = data.apply(major_land_cover, axis=1)

        # Merge the decoded data with the shapefile
        merged_shapefile = county_shapefile.merge(data[['STATEFP', 'COUNTYFP', 'major_land_cover']], on=['STATEFP', 'COUNTYFP'], how='left')

        # Plot the map
        fig = plt.figure(figsize=(15, 10))
        ax = fig.add_axes([0.1, 0.1, 0.8, 0.8], projection=ccrs.PlateCarree())
        ax.set_extent([-125, -66.5, 24, 49.5], crs=ccrs.PlateCarree())  # Continental US extent

        # Add base map features
        ax.add_feature(cartopy.feature.LAND)
        ax.add_feature(cartopy.feature.OCEAN)
        ax.add_feature(cartopy.feature.COASTLINE)
        ax.add_feature(cartopy.feature.BORDERS, linestyle=':')

        # Add state boundaries for better context
        ax.add_feature(cartopy.feature.STATES, edgecolor='gray')

        # Plot major land cover
        land_cover_colors = {
            'Developed': 'grey',
            'Cropland': 'yellow',
            'Grass/Shrub': 'green',
            'Water': 'blue',
            'Wetland': 'cyan',
            'Ice/Snow': 'white',
            'Barren': 'brown',
            'Unknown': 'black'
        }
        merged_shapefile['color'] = merged_shapefile['major_land_cover'].map(land_cover_colors)

        merged_shapefile.plot(ax=ax, color=merged_shapefile['color'], edgecolor='0.8', linewidth=0.8, transform=ccrs.PlateCarree(), alpha=0.75)

        # Add legend
        handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label)
                   for label, color in land_cover_colors.items()]
        ax.legend(handles=handles, title='Major Adjunct Land Cover', loc='lower left', bbox_to_anchor=(1, 0))

        plt.title(f'Major Adjunct Land Cover by County ({year})')
        plt.savefig(f'{output_dir}/adjunct_landcover_{year}.png', dpi=100, bbox_inches='tight')
        plt.close()

    else:
        print(f"File not found for the year {year}")

print("Major adjunct land cover maps created successfully!")

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy
import os
from collections import Counter
from tqdm import trange

# Define file paths
csv_base_path = "G:\\Hangkai\\CONUS_Forest_Edge_LCMAP\\Edge_adjunct_LC\\Ecoregion_Classification\\county_{}.csv"
shapefile_path = "G:\\Hangkai\\CONUS Forest Edge Mapping\\CONUS shapefile\\cb_2018_us_county_5m.shp"
output_dir = "adjunct_landcover_maps"

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Read the Shapefile
county_shapefile = gpd.read_file(shapefile_path)

# Convert key columns to string to ensure proper merging
county_shapefile['STATEFP'] = county_shapefile['STATEFP'].astype(str).str.zfill(2)
county_shapefile['COUNTYFP'] = county_shapefile['COUNTYFP'].astype(str).str.zfill(3)

# Define land cover mapping
land_cover_mapping = {
    '1': 'Developed',
    '2': 'Cropland',
    '3': 'Grass/Shrub',
    '0': 'Tree Cover',  # Recoded from '4'
    '5': 'Water',
    '6': 'Wetland',
    '7': 'Ice/Snow',
    '8': 'Barren'
}

def decode_landcover(code):
    """ Decode the five-digit landcover code into readable labels for each side. """
    code_str = str(int(code)).zfill(5)
    return [
        land_cover_mapping.get(digit, 'Unknown') for digit in code_str[1:] if digit != '0'
    ]

# Function to determine the major land cover type
def major_land_cover(row):
    counter = Counter()
    total_forest_pixels = sum(row[col] for col in row.index if col.startswith('1'))
    if total_forest_pixels == 0:
        return 'No Forest'
    for col in row.index:
        if col not in ['STATEFP', 'NAME', 'COUNTYFP']:
            land_covers = decode_landcover(col)
            pixel_sum = row[col]
            land_cover_count = Counter(land_covers)
            for land_cover, count in land_cover_count.items():
                if land_cover != 'Tree Cover':
                    counter[land_cover] += pixel_sum * count
    if counter:
        return max(counter, key=counter.get)
    return 'Unknown'

# Loop through each year's data and generate maps
for year in trange(1985, 2022):
    print(f'Processing year: {year}')
    csv_path = csv_base_path.format(year)
    
    if os.path.exists(csv_path):
        data = pd.read_csv(csv_path)

        # Convert key columns to string to ensure proper merging
        data['STATEFP'] = data['STATEFP'].astype(str).str.zfill(2)
        data['COUNTYFP'] = data['COUNTYFP'].astype(str).str.zfill(3)

        # Fill NaN values with 0
        data = data.fillna(0)

        # Decode the landcover and find the major landcover type for each county
        data['major_land_cover'] = data.apply(major_land_cover, axis=1)

        # Merge the decoded data with the shapefile
        merged_shapefile = county_shapefile.merge(data[['STATEFP', 'COUNTYFP', 'major_land_cover']], on=['STATEFP', 'COUNTYFP'], how='left')

        # Plot the map
        fig = plt.figure(figsize=(15, 10))
        ax = fig.add_axes([0.1, 0.1, 0.8, 0.8], projection=ccrs.PlateCarree())
        ax.set_extent([-125, -66.5, 24, 49.5], crs=ccrs.PlateCarree())  # Continental US extent

        # Add base map features
        ax.add_feature(cartopy.feature.LAND)
        ax.add_feature(cartopy.feature.OCEAN)
        ax.add_feature(cartopy.feature.COASTLINE)
        ax.add_feature(cartopy.feature.BORDERS, linestyle=':')

        # Add state boundaries for better context
        ax.add_feature(cartopy.feature.STATES, edgecolor='gray')

        # Plot major land cover
        land_cover_colors = {
            'Developed': 'grey',
            'Cropland': 'yellow',
            'Grass/Shrub': 'green',
            'Water': 'blue',
            'Wetland': 'cyan',
            'Ice/Snow': 'white',
            'Barren': 'brown',
            'No Forest': 'black',
        }
        merged_shapefile['color'] = merged_shapefile['major_land_cover'].map(land_cover_colors)

        merged_shapefile.plot(ax=ax, color=merged_shapefile['color'], edgecolor='0.8', linewidth=0.8, transform=ccrs.PlateCarree(), alpha=0.75)

        # Add legend
        handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=10, label=label)
                   for label, color in land_cover_colors.items()]
        ax.legend(handles=handles, title='Major Adjunct Land Cover', loc='lower left', bbox_to_anchor=(1, 0))

        plt.title(f'Major Adjunct Land Cover by County ({year})')
        plt.savefig(f'{output_dir}/adjunct_landcover_{year}.png', dpi=100, bbox_inches='tight')
        plt.close()

    else:
        print(f"File not found for the year {year}")

print("Major adjunct land cover maps created successfully!")

In [ ]:
from PIL import Image
import os

# File path to the images
image_folder = 'adjunct_landcover_maps'
images = []

# Loop through the years and load images
for year in range(1985, 2022):
    file_path = os.path.join(image_folder, f'adjunct_landcover_{year}.png')
    images.append(Image.open(file_path))

# Save images as GIF
gif_path = 'adjunct_landcover_maps_v0.gif'
images[0].save(
    gif_path,
    save_all=True,
    append_images=images[1:],
    duration=200,  # Duration in milliseconds
    loop=0,
    optimize=False,  # Disable optimization
    disposal=2  # Restore to background color
)

print("GIF created successfully!")